# **Loading Dataset**

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('train.txt', sep = ";", header=None, names=['text', 'emotion'])

In [5]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [6]:
df.isnull().sum()

,0
text,0
emotion,0


# **Cleaning Dataset**

1) Convert Emotions into Number

In [7]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emotions:
    emotion_numbers[emotion] = i
    i += 1

df.emotion = df.emotion.map(emotion_numbers)

In [8]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


2) Convert text into Lowercase

In [9]:
df['text'] = df['text'].apply(lambda X: X.lower())

In [10]:
import string

def remove_pun(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_pun)

3) Remove Numbers from text

In [11]:
def remove_number(text):
  new = ""
  for i in text:
    if not i.isdigit():
      new = new + i
  return new

df['text'] = df['text'].apply(remove_number)

4. Remove Emojis from text

In [12]:
def remove_emojis(text):
  new = ""
  for i in text:
    if i.isascii():
      new = new + i
  return new

df['text'] = df['text'].apply(remove_emojis)

5. Removing Stop words

In [13]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [14]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [15]:
def remove_stop_words(txt):
  words = txt.split()
  cleaned = []
  for word in words:
    if word not in stop_words:
      cleaned.append(word)
  return " ".join(cleaned)

df['text'] = df['text'].apply(remove_stop_words)

In [16]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

# **Spliting Data and Vectorize the Dataset**

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [19]:
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
bow_transformer = CountVectorizer()
X_train_bow = bow_transformer.fit_transform(X_train)
X_test_bow = bow_transformer.transform(X_test)

# **Model Training**

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nv_model = MultinomialNB()
nv_model.fit(X_train_bow, y_train)
y_pred = nv_model.predict(X_test_bow)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.95      0.84       946
           1       0.89      0.63      0.74       427
           2       0.93      0.27      0.41       296
           3       1.00      0.05      0.10       113
           4       0.85      0.57      0.68       397
           5       0.73      0.96      0.83      1021

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200

[[899   8   3   0   4  32]
 [ 87 271   0   0  13  56]
 [ 51   6  79   0   4 156]
 [ 40   0   1   6  17  49]
 [ 87  15   0   0 226  69]
 [ 36   4   2   0   2 977]]
Accuracy: 0.768125


In [23]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)
y_pred = nb2_model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.93      0.80       946
           1       0.93      0.29      0.44       427
           2       1.00      0.03      0.06       296
           3       1.00      0.01      0.02       113
           4       0.92      0.22      0.36       397
           5       0.60      0.99      0.74      1021

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200

[[ 883    0    0    0    2   61]
 [ 145  123    0    0    2  157]
 [  48    1    9    0    1  237]
 [  41    0    0    1    3   68]
 [ 137    8    0    0   88  164]
 [  10    0    0    0    0 1011]]
Accuracy: 0.6609375


In [25]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [26]:
log_pred = logistic_model.predict(X_test_tfidf)
print(classification_report(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))
print("Accuracy:", accuracy_score(y_test, log_pred))

              precision    recall  f1-score   support

           0       0.90      0.94      0.92       946
           1       0.90      0.81      0.86       427
           2       0.90      0.61      0.73       296
           3       0.88      0.47      0.61       113
           4       0.86      0.76      0.81       397
           5       0.81      0.96      0.88      1021

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200

[[893  13   1   0   7  32]
 [ 30 347   1   0  12  37]
 [ 12   5 182   0   4  93]
 [ 15   0   1  53  24  20]
 [ 24  17   1   7 302  46]
 [ 15   2  17   0   3 984]]
Accuracy: 0.8628125
